# Graph Orchestration and Memory Test

Validates LangGraph state creation, graph compile, checkpointer setup, and optional end-to-end graph invocation.

In [ ]:
from pathlib import Path
import sys
import time

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src').exists()), cwd)
sys.path.insert(0, str(project_root / 'src'))
print('project_root:', project_root)

In [ ]:
from graph.state import initial_state

state = initial_state(
    query='Why did retention drop last month?',
    thread_id='notebook-graph-test',
    user_id='notebook-user',
    time_range='last_month',
)

for key in sorted(state):
    print(f'{key}: {state[key]}')

assert state['query']
assert state['thread_id'] == 'notebook-graph-test'
assert state['agent_results'] == []
assert state['guardrail_passed'] is True

In [ ]:
from memory.checkpointer import get_checkpointer

checkpointer = get_checkpointer()
print('checkpointer:', checkpointer)
assert checkpointer is not None

In [ ]:
from graph.graph import copilot_graph, route_after_pre_hook, route_to_agents

print('copilot_graph:', type(copilot_graph))
assert copilot_graph is not None

blocked = dict(state)
blocked['guardrail_passed'] = False
assert route_after_pre_hook(blocked) == 'post_hook'
assert route_after_pre_hook(state) == 'supervisor'

empty_route = dict(state)
empty_route['next_agents'] = []
assert route_to_agents(empty_route) == 'synthesizer'
print('routing helpers ok')

## Optional Full Graph Query

Set `RUN_FULL_GRAPH = True` to invoke the full graph. This can call the configured LLM provider for intent classification and synthesis.

In [ ]:
RUN_FULL_GRAPH = False

if RUN_FULL_GRAPH:
    config = {'configurable': {'thread_id': state['thread_id']}}
    t0 = time.time()
    result = copilot_graph.invoke(state, config=config)
    elapsed_ms = round((time.time() - t0) * 1000, 2)
    print('elapsed_ms:', elapsed_ms)
    print('intent:', result.get('intent'))
    print('agents:', [r.get('agent') for r in result.get('agent_results', [])])
    print('summary:', result.get('final_summary', '')[:1000])
    assert result.get('final_summary')
else:
    print('Skipped. Set RUN_FULL_GRAPH = True to run this cell.')